# Pattern / location / scale similarity of two images

Trains a Siamese network on **Oxford-IIIT Pet** (37 cat & dog breeds, with segmentation masks). Given two images it returns

| score | what it measures | identical images |
|---|---|---|
| **pattern** | is it the same thing? (same animal 1, same breed 0.75, same species 0.4, cat vs dog 0) | 1 |
| **location** | `1 / (1 + d / 0.1)`, `d` = distance between the objects' centres (fraction of image size) | 1 |
| **scale** | smaller size / larger size | 1 |
| **overall** | geometric mean of the three | 1 |

plus the raw displacement `dx, dy` and `scale_ratio` of B relative to A.

**Runtime → Change runtime type → T4 GPU** before running. Rough timing on a T4: download + preprocessing a few minutes, training ~15–20 min.

In [ ]:
# ---- settings ----
GITHUB_USER = "your-github-id"   # <- your GitHub ID once the repo is pushed (or upload pattern-sim.zip to /content)
REPO = "pattern-sim"
USE_DRIVE = True                 # keep the preprocessed data + checkpoints in Google Drive

import os, sys, subprocess
REPO_DIR = f"/content/{REPO}"
if not os.path.isdir(REPO_DIR):
    if os.path.exists(f"/content/{REPO}.zip"):
        subprocess.run(["unzip", "-q", f"/content/{REPO}.zip", "-d", "/content"], check=True)
    else:
        subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO}.git", REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/pattern-sim"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

In [ ]:
# 30-second self-test on fake data (no download) - catches setup problems early
!python -m pytest -q tests

## 1. Data
Each photo becomes a square RGBA *tile* (alpha = the animal's mask). During training, pairs are rendered on the GPU: each animal is placed at a random centre and size, either cut out onto a random background or inside its original photo. Because we do the shifting/resizing ourselves, the true location and size labels are exact. The green boxes below are those labels.

In [ ]:
from patsim.data import build_bank, PairSynth
from patsim.viz import show_batch

DEV = "cuda"
train_bank = build_bank("/content/data", "trainval", tile=160, cache_dir=f"{WORK}/cache").to(DEV)
test_bank = build_bank("/content/data", "test", tile=160, cache_dir=f"{WORK}/cache").to(DEV)
print(len(train_bank), "train /", len(test_bank), "test animals,", len(train_bank.classes), "breeds")

show_batch(PairSynth(train_bank, canvas=160).sample(8), n=8);

## 2. Train
Every `eval_every` steps it scores 1024 fixed pairs built from the **test** split (animals never seen in training) and saves `patsim.pt` to `WORK/runs/r18`.

In [ ]:
from patsim.train import Config, train

cfg = Config(out_dir=f"{WORK}/runs/r18", steps=6000, batch_size=64)
model, history = train(cfg, train_bank, test_bank)

In [ ]:
import pandas as pd

h = pd.DataFrame(history).set_index("step")
display(h.round(3).T)
h[["pattern_mae", "location_mae", "scale_mae"]].plot(marker="o", title="test-pair error vs. true score (lower is better)");

## 3. Your scenarios
A cat at the top-left compared with: itself, itself moved to the bottom-right, itself bigger, a dog at the same place, another cat of the same breed, and itself inside its original photo.

In [ ]:
from patsim.infer import PairScorer
from patsim.viz import show_result

scorer = PairScorer.from_checkpoint(f"{WORK}/runs/r18/patsim.pt")
test_synth = PairSynth(test_bank, canvas=160)
g = torch.Generator(device=DEV).manual_seed(0)

def put(i, center, size, photo=False):
    """Render test animal i with its box centre at `center` (fractions of the image) and box size `size`."""
    img, _ = test_synth.render([i], [center], [size], photo=photo, g=g)
    return img[0]

breed, species = test_bank.breed.cpu(), test_bank.species.cpu()
cat = int((species == 0).nonzero()[0])
cat2 = int(((breed == breed[cat]) & (torch.arange(len(breed)) != cat)).nonzero()[0])
dog = int((species == 1).nonzero()[0])
TL, BR = (0.33, 0.33), (0.67, 0.67)

cat_tl = put(cat, TL, 0.4)
cases = {
    "identical picture": (cat_tl, cat_tl),
    "same cat, moved to bottom-right": (cat_tl, put(cat, BR, 0.4)),
    "same cat, 1.6x bigger": (cat_tl, put(cat, TL, 0.64)),
    "dog, same place and size": (cat_tl, put(dog, TL, 0.4)),
    "another cat, same breed": (cat_tl, put(cat2, TL, 0.4)),
    "same cat in its original photo, bottom-right": (cat_tl, put(cat, BR, 0.4, photo=True)),
}
rows = {}
for name, (a, b) in cases.items():
    r = scorer.score(a, b, return_maps=True)
    show_result(r, title=name)
    rows[name] = {k: round(r[k], 3) for k in ("pattern", "location", "scale", "overall", "dx", "dy", "scale_ratio")}
display(pd.DataFrame(rows).T)

In [ ]:
# Two raw, unmodified test photos
from torchvision.datasets import OxfordIIITPet

pets = OxfordIIITPet("/content/data", split="test")
show_result(scorer.score(pets[0][0], pets[1][0], return_maps=True), title="two raw test photos");

In [ ]:
# Your own images: pick two files
from google.colab import files

up = list(files.upload().keys())
r = scorer.score(up[0], up[1], return_maps=True)
show_result(r, title=" vs ".join(up))
{k: v for k, v in r.items() if k != "maps"}

## Knobs
* **Score shapes** (no retraining): `PairScorer.from_checkpoint(path, d0=0.1, tau=1.0)` - `d0` is the displacement at which location = 0.5; `tau` softens the scale score.
* **What "same pattern" means** (retrain): `Config(pattern_levels=(1.0, 0.75, 0.4, 0.0))` for same animal / breed / species / other species.
* **Object sizes seen in training** (retrain): `Config(size_range=(0.25, 0.7))`, as a fraction of the image side.
* **Longer training** usually helps the pattern score most: `Config(steps=15000)`.